# Successor Long-Term Phase Plan

This notebook is the inspectable planning companion to `docs/plans/0001_successor_roadmap.md` and `docs/plans/0005_v1_capability_parity_matrix.md`.
It does two things:

1. shows the current proved state directly from the repo;
2. lays out the completed bootstrap phases plus the completed and remaining post-bootstrap parity phases.

The future-phase code cells are intentionally design-oriented. They print pseudocode or provisional
contract sketches so you can run the notebook and see where the repo is going even before the supporting
implementation exists.

In [1]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from pathlib import Path
import json
import subprocess
import textwrap

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

def git(*args: str) -> str:
    return subprocess.check_output(["git", "-C", str(repo_root), *args], text=True).strip()

def bullet_list(items: tuple[str, ...] | list[str]) -> str:
    return "\n".join(f"- {item}" for item in items)

def heading(title: str) -> None:
    print(f"\n{title}\n" + "=" * len(title))

@dataclass(frozen=True)
class PhasePlan:
    phase_id: str
    title: str
    status: str
    confidence: str
    goal: str
    why_now: tuple[str, ...] = field(default_factory=tuple)
    build: tuple[str, ...] = field(default_factory=tuple)
    acceptance: tuple[str, ...] = field(default_factory=tuple)
    evidence: tuple[str, ...] = field(default_factory=tuple)
    dependencies: tuple[str, ...] = field(default_factory=tuple)
    notes: tuple[str, ...] = field(default_factory=tuple)

def show_phase(phase: PhasePlan) -> None:
    heading(f"Phase {phase.phase_id}: {phase.title}")
    print(f"status: {phase.status}")
    print(f"confidence: {phase.confidence}")
    print(f"goal: {phase.goal}")
    if phase.why_now:
        print("\nwhy now:\n" + bullet_list(phase.why_now))
    if phase.build:
        print("\nbuild:\n" + bullet_list(phase.build))
    if phase.acceptance:
        print("\nacceptance criteria:\n" + bullet_list(phase.acceptance))
    if phase.evidence:
        print("\nacceptance evidence:\n" + bullet_list(phase.evidence))
    if phase.dependencies:
        print("\ndependencies:\n" + bullet_list(phase.dependencies))
    if phase.notes:
        print("\nnotes:\n" + bullet_list(phase.notes))

In [2]:
current_snapshot = {
    "current_commit": git("rev-parse", "--short", "HEAD"),
    "current_branch": git("branch", "--show-current"),
    "tracked_notebooks": sorted(path.name for path in (repo_root / "notebooks").glob("*.ipynb")),
    "prompt_assets": sorted(str(path.relative_to(repo_root)) for path in (repo_root / "prompts").rglob("*.yaml")),
    "test_modules": sorted(str(path.relative_to(repo_root)) for path in (repo_root / "tests").rglob("test_*.py")),
    "roadmap": str((repo_root / "docs" / "plans" / "0001_successor_roadmap.md").relative_to(repo_root)),
    "parity_matrix": str((repo_root / "docs" / "plans" / "0005_v1_capability_parity_matrix.md").relative_to(repo_root)),
    "status_doc": str((repo_root / "docs" / "STATUS.md").relative_to(repo_root)),
}

heading("Repo Snapshot")
print(json.dumps(current_snapshot, indent=2))


Repo Snapshot
{
  "current_commit": "75a74df",
  "current_branch": "main",
  "tracked_notebooks": [
    "00_master_governed_text_to_reviewed_assertions.ipynb",
    "01_policy_contracts.ipynb",
    "02_donor_profile_loading.ipynb",
    "03_validation_slice.ipynb",
    "04_review_slice.ipynb",
    "05_review_reporting_slice.ipynb",
    "06_overlay_application_slice.ipynb",
    "07_text_grounded_import_contract.ipynb",
    "08_text_extraction_slice.ipynb",
    "09_successor_long_term_plan.ipynb",
    "10_live_extraction_evaluation.ipynb",
    "11_future_phase_breakdown.ipynb",
    "12_cli_surface.ipynb",
    "13_dodaf_minimal_second_pack.ipynb",
    "14_artifact_lineage_slice.ipynb",
    "15_epistemic_extension_slice.ipynb",
    "16_governed_bundle_workflow.ipynb",
    "17_canonical_graph_recovery_slice.ipynb",
    "18_stable_identity_slice.ipynb",
    "19_semantic_canonicalization_slice.ipynb",
    "20_whygame_mcp_slice.ipynb"
  ],
  "prompt_assets": [
    "prompts/evaluation/judge_cand

In [3]:
completed_phases = [
    PhasePlan(
        phase_id='0',
        title='Ontology Runtime Slice',
        status='completed',
        confidence='high',
        goal='Prove ontology/runtime contracts, donor loading, policy semantics, and local validation.',
        evidence=(
            'tests/ontology_runtime/',
            'notebooks/01_policy_contracts.ipynb',
            'notebooks/02_donor_profile_loading.ipynb',
            'notebooks/03_validation_slice.ipynb',
        ),
    ),
    PhasePlan(
        phase_id='1',
        title='Pipeline Review Slice',
        status='completed',
        confidence='high',
        goal='Persist validation, proposals, and configurable mixed-mode review state.',
        evidence=(
            'tests/pipeline/test_review_service.py',
            'notebooks/04_review_slice.ipynb',
        ),
    ),
    PhasePlan(
        phase_id='2',
        title='First User-Visible Capability',
        status='completed',
        confidence='high',
        goal='Recover governed review of candidate assertions as the first actual payoff.',
        evidence=(
            'src/onto_canon6/surfaces/review_report.py',
            'notebooks/05_review_reporting_slice.ipynb',
        ),
    ),
    PhasePlan(
        phase_id='3',
        title='Overlay Application and Query Surface',
        status='completed',
        confidence='high',
        goal='Make accepted ontology-growth decisions operational and inspectable.',
        evidence=(
            'src/onto_canon6/pipeline/overlay_service.py',
            'notebooks/06_overlay_application_slice.ipynb',
        ),
    ),
    PhasePlan(
        phase_id='4',
        title='Text-Grounded Producer Integration',
        status='completed',
        confidence='high',
        goal='Route raw text through llm_client into evidence-grounded candidate assertions.',
        evidence=(
            'src/onto_canon6/pipeline/text_extraction.py',
            'prompts/extraction/text_to_candidate_assertions.yaml',
            'notebooks/07_text_grounded_import_contract.ipynb',
            'notebooks/08_text_extraction_slice.ipynb',
        ),
    ),
    PhasePlan(
        phase_id='5',
        title='Live Extraction Evaluation and Quality Harness',
        status='completed',
        confidence='high',
        goal='Measure real extraction usefulness before expanding the workflow further.',
        evidence=(
            'src/onto_canon6/evaluation/service.py',
            'prompts/evaluation/judge_candidate_reasonableness.yaml',
            'tests/evaluation/test_service.py',
            'notebooks/10_live_extraction_evaluation.ipynb',
            'docs/adr/0005-separate-live-extraction-reasonableness-from-structural-validation-and-canonicalization-fidelity.md',
        ),
        notes=(
            'This phase is now complete and is the guardrail for future extraction work.',
        ),
    ),
    PhasePlan(
        phase_id='6',
        title='First Operational Surface',
        status='completed',
        confidence='high',
        goal='Make the workflow usable without direct Python or notebook editing.',
        evidence=(
            'src/onto_canon6/cli.py',
            'tests/integration/test_cli_flow.py',
            'notebooks/12_cli_surface.ipynb',
            'docs/adr/0006-prefer-cli-as-the-first-operational-surface-before-mcp-or-ui.md',
        ),
        notes=(
            'The first operational proof is now real and remains deliberately CLI-first.',
        ),
    ),
    PhasePlan(
        phase_id='7',
        title='Domain Pack Generalization',
        status='completed',
        confidence='high',
        goal='Prove the same architecture supports a second real domain pack through local pack and profile data.',
        evidence=(
            'ontology_packs/dodaf_minimal/0.1.0/manifest.yaml',
            'profiles/dodaf_minimal_strict/0.1.0/manifest.yaml',
            'profiles/dodaf_minimal_mixed/0.1.0/manifest.yaml',
            'tests/ontology_runtime/test_dodaf_minimal.py',
            'tests/integration/test_dodaf_minimal_cli.py',
            'notebooks/13_dodaf_minimal_second_pack.ipynb',
        ),
        notes=(
            'One pack now supports strict and mixed profiles without core branching.',
        ),
    ),
    PhasePlan(
        phase_id='8',
        title='Artifact Lineage Recovery',
        status='completed',
        confidence='high',
        goal='Recover artifact-backed provenance through a narrow three-kind model without rebuilding a fused runtime.',
        evidence=(
            'src/onto_canon6/artifacts/',
            'src/onto_canon6/surfaces/lineage_report.py',
            'tests/artifacts/test_artifact_service.py',
            'notebooks/14_artifact_lineage_slice.ipynb',
            'docs/adr/0008-start-artifact-lineage-with-a-narrow-three-kind-model-and-candidate-centered-links.md',
        ),
        notes=(
            'Artifact lineage now stays bounded, candidate-centered, and traversal-based first.',
        ),
    ),
    PhasePlan(
        phase_id='9',
        title='Epistemic Extension',
        status='completed',
        confidence='high',
        goal='Add extension-local confidence and supersession over accepted candidates without changing the base review schema.',
        evidence=(
            'src/onto_canon6/extensions/epistemic/',
            'src/onto_canon6/surfaces/epistemic_report.py',
            'tests/extensions/test_epistemic_service.py',
            'notebooks/15_epistemic_extension_slice.ipynb',
            'docs/adr/0009-start-epistemic-extension-with-confidence-and-supersession-over-accepted-candidates.md',
        ),
    ),
    PhasePlan(
        phase_id='10',
        title='Product-Facing Workflow Integration',
        status='completed',
        confidence='high',
        goal='Export a CLI-driven governed bundle so the successor ends in a real downstream artifact.',
        evidence=(
            'src/onto_canon6/surfaces/governed_bundle.py',
            'src/onto_canon6/cli.py',
            'tests/surfaces/test_governed_bundle.py',
            'tests/integration/test_cli_flow.py',
            'notebooks/16_governed_bundle_workflow.ipynb',
            'docs/adr/0010-choose-cli-driven-governed-bundle-export-as-the-first-product-facing-workflow.md',
        ),
        notes=(
            'The first outward-facing boundary stays CLI-backed and exports a governed JSON bundle instead of adding MCP too early.',
        ),
    ),
    PhasePlan(
        phase_id='11',
        title='Canonical Graph Recovery',
        status='completed',
        confidence='high',
        goal='Recover the durable concept/entity and assertion/belief graph from accepted candidates.',
        evidence=(
            'src/onto_canon6/core/graph_service.py',
            'tests/core/test_graph_service.py',
            'tests/integration/test_graph_cli.py',
            'notebooks/17_canonical_graph_recovery_slice.ipynb',
            'docs/adr/0012-start-canonical-graph-recovery-with-explicit-promotion-from-accepted-candidates.md',
        ),
    ),
    PhasePlan(
        phase_id='12',
        title='Stable Identity and External References',
        status='completed',
        confidence='high',
        goal='Recover stable identities, aliases, and explicit external-reference handling.',
        evidence=(
            'src/onto_canon6/core/identity_service.py',
            'tests/core/test_identity_service.py',
            'tests/integration/test_identity_cli.py',
            'notebooks/18_stable_identity_slice.ipynb',
            'docs/adr/0013-start-stable-identity-with-promoted-entity-identities-alias-membership-and-explicit-external-reference-state.md',
        ),
    ),
    PhasePlan(
        phase_id='13',
        title='Semantic Canonicalization Stack Recovery Or Replacement',
        status='completed',
        confidence='high',
        goal='Replace the hard v1 semantic stack with pack-driven promoted-assertion canonicalization and explicit recanonicalization.',
        evidence=(
            'src/onto_canon6/core/semantic_service.py',
            'tests/core/test_semantic_service.py',
            'tests/integration/test_semantic_cli.py',
            'notebooks/19_semantic_canonicalization_slice.ipynb',
            'docs/adr/0014-replace-the-v1-semantic-stack-with-pack-driven-canonicalization-and-explicit-recanonicalization.md',
        ),
    ),
]
completed_phases.append(
    PhasePlan(
        phase_id='14',
        title='Agent Surface And Adapter Recovery',
        status='completed',
        confidence='high',
        goal='Recover one richer external surface and one real adapter without rebuilding the v1 monolith.',
        evidence=(
            'src/onto_canon6/mcp_server.py',
            'src/onto_canon6/adapters/whygame_service.py',
            'tests/integration/test_mcp_server.py',
            'notebooks/20_whygame_mcp_slice.ipynb',
            'docs/adr/0015-recover-phase-14-through-a-thin-mcp-surface-and-a-whygame-relationship-adapter.md',
        ),
    ),
)
planned_phases = [
    PhasePlan(
        phase_id='15',
        title='Broader Epistemics, Corroboration, And Temporal/Inference Recovery',
        status='planned',
        confidence='low',
        goal='Recover the broader v1 belief-management behaviors only after the graph, identity, semantic, and agent-surface layers exist.',
        evidence=(
            'docs/plans/0001_successor_roadmap.md',
            'docs/plans/0005_v1_capability_parity_matrix.md',
            'notebooks/11_future_phase_breakdown.ipynb',
        ),
    ),
]

heading('Completed Phases')
for phase in completed_phases:
    print(f"Phase {phase.phase_id}: {phase.title} [{phase.status}]")
    print(f"  goal: {phase.goal}")
    print(f"  evidence count: {len(phase.evidence)}")

heading('Planned Phases')
for phase in planned_phases:
    print(f"Phase {phase.phase_id}: {phase.title} [{phase.status}]")
    print(f"  goal: {phase.goal}")
    print(f"  evidence count: {len(phase.evidence)}")



Completed Phases
Phase 0: Ontology Runtime Slice [completed]
  goal: Prove ontology/runtime contracts, donor loading, policy semantics, and local validation.
  evidence count: 4
Phase 1: Pipeline Review Slice [completed]
  goal: Persist validation, proposals, and configurable mixed-mode review state.
  evidence count: 2
Phase 2: First User-Visible Capability [completed]
  goal: Recover governed review of candidate assertions as the first actual payoff.
  evidence count: 2
Phase 3: Overlay Application and Query Surface [completed]
  goal: Make accepted ontology-growth decisions operational and inspectable.
  evidence count: 2
Phase 4: Text-Grounded Producer Integration [completed]
  goal: Route raw text through llm_client into evidence-grounded candidate assertions.
  evidence count: 4
Phase 5: Live Extraction Evaluation and Quality Harness [completed]
  goal: Measure real extraction usefulness before expanding the workflow further.
  evidence count: 5
Phase 6: First Operational Surfac

## Sequence Status

The bootstrap roadmap is fully proved through Phase 10. Phase 11 is proved as the first bounded canonical-graph recovery slice, Phase 12 is proved as the first bounded stable-identity slice, Phase 13 is proved as the first bounded semantic canonicalization replacement slice, and Phase 14 is now proved as the thin MCP plus WhyGame adapter recovery slice. The broader successor now continues through the explicit parity matrix and the planned Phase 15 extension.

In [4]:
recommended_sequence = [
    ("8", "Recover artifact lineage through a narrow bounded subsystem."),
    ("9", "Add epistemic reasoning as an extension instead of hidden policy."),
    ("10", "Export one real governed bundle through the CLI-backed workflow."),
]

heading("Bootstrap Sequence")
for phase_id, rationale in recommended_sequence:
    print(f"Phase {phase_id}: {rationale}")



Bootstrap Sequence
Phase 8: Recover artifact lineage through a narrow bounded subsystem.
Phase 9: Add epistemic reasoning as an extension instead of hidden policy.
Phase 10: Export one real governed bundle through the CLI-backed workflow.


## Phase 5 Completed Slice


In [5]:
phase = next(item for item in completed_phases if item.phase_id == '5')
show_phase(phase)

print("\nas_json:")
print(json.dumps(asdict(phase), indent=2))



Phase 5: Live Extraction Evaluation and Quality Harness
status: completed
confidence: high
goal: Measure real extraction usefulness before expanding the workflow further.

acceptance evidence:
- src/onto_canon6/evaluation/service.py
- prompts/evaluation/judge_candidate_reasonableness.yaml
- tests/evaluation/test_service.py
- notebooks/10_live_extraction_evaluation.ipynb
- docs/adr/0005-separate-live-extraction-reasonableness-from-structural-validation-and-canonicalization-fidelity.md

notes:
- This phase is now complete and is the guardrail for future extraction work.

as_json:
{
  "phase_id": "5",
  "title": "Live Extraction Evaluation and Quality Harness",
  "status": "completed",
  "confidence": "high",
  "goal": "Measure real extraction usefulness before expanding the workflow further.",
  "why_now": [],
  "build": [],
  "acceptance": [],
  "evidence": [
    "src/onto_canon6/evaluation/service.py",
    "prompts/evaluation/judge_candidate_reasonableness.yaml",
    "tests/evaluation

In [6]:
implemented_assets = (
    "src/onto_canon6/evaluation/models.py",
    "src/onto_canon6/evaluation/service.py",
    "prompts/evaluation/judge_candidate_reasonableness.yaml",
    "tests/evaluation/test_service.py",
    "tests/fixtures/psyop_eval_slice.json",
    "notebooks/10_live_extraction_evaluation.ipynb",
    "docs/adr/0005-separate-live-extraction-reasonableness-from-structural-validation-and-canonicalization-fidelity.md",
)
heading("Implemented Slice")
print(bullet_list(implemented_assets))



Implemented Slice
- src/onto_canon6/evaluation/models.py
- src/onto_canon6/evaluation/service.py
- prompts/evaluation/judge_candidate_reasonableness.yaml
- tests/evaluation/test_service.py
- tests/fixtures/psyop_eval_slice.json
- notebooks/10_live_extraction_evaluation.ipynb
- docs/adr/0005-separate-live-extraction-reasonableness-from-structural-validation-and-canonicalization-fidelity.md


## Phase 6 Completed Slice

In [7]:
phase = next(item for item in completed_phases if item.phase_id == '6')
show_phase(phase)

print('\nas_json:')
print(json.dumps(asdict(phase), indent=2))



Phase 6: First Operational Surface
status: completed
confidence: high
goal: Make the workflow usable without direct Python or notebook editing.

acceptance evidence:
- src/onto_canon6/cli.py
- tests/integration/test_cli_flow.py
- notebooks/12_cli_surface.ipynb
- docs/adr/0006-prefer-cli-as-the-first-operational-surface-before-mcp-or-ui.md

notes:
- The first operational proof is now real and remains deliberately CLI-first.

as_json:
{
  "phase_id": "6",
  "title": "First Operational Surface",
  "status": "completed",
  "confidence": "high",
  "goal": "Make the workflow usable without direct Python or notebook editing.",
  "why_now": [],
  "build": [],
  "acceptance": [],
  "evidence": [
    "src/onto_canon6/cli.py",
    "tests/integration/test_cli_flow.py",
    "notebooks/12_cli_surface.ipynb",
    "docs/adr/0006-prefer-cli-as-the-first-operational-surface-before-mcp-or-ui.md"
  ],
  "dependencies": [],
  "notes": [
    "The first operational proof is now real and remains deliberately

In [8]:
implemented_workflow = textwrap.dedent(r"""
# Implemented product-facing slice:
# 1. ingest one source text file through the CLI
# 2. extract candidate assertions
# 3. review candidates and ontology proposals
# 4. apply ontology additions where approved
# 5. export one governed JSON bundle with provenance and governance state

workflow_demo = {
    "input": "real source text file",
    "surface": "CLI-backed governed bundle export",
    "outputs": [
        "accepted candidate assertions",
        "linked proposals and overlay applications",
        "source provenance and evidence spans",
        "candidate artifact lineage when present",
        "candidate epistemic state when present",
        "summary counts for downstream inspection",
    ],
}
""").strip()
heading("Implemented Workflow")
print(implemented_workflow)



Implemented Workflow
# Implemented product-facing slice:
# 1. ingest one source text file through the CLI
# 2. extract candidate assertions
# 3. review candidates and ontology proposals
# 4. apply ontology additions where approved
# 5. export one governed JSON bundle with provenance and governance state

workflow_demo = {
    "input": "real source text file",
    "surface": "CLI-backed governed bundle export",
    "outputs": [
        "accepted candidate assertions",
        "linked proposals and overlay applications",
        "source provenance and evidence spans",
        "candidate artifact lineage when present",
        "candidate epistemic state when present",
        "summary counts for downstream inspection",
    ],
}


## Phase 7 Retrospective

In [9]:
phase = next(item for item in completed_phases if item.phase_id == '7')
show_phase(phase)

print("\nas_json:")
print(json.dumps(asdict(phase), indent=2))



Phase 7: Domain Pack Generalization
status: completed
confidence: high
goal: Prove the same architecture supports a second real domain pack through local pack and profile data.

acceptance evidence:
- ontology_packs/dodaf_minimal/0.1.0/manifest.yaml
- profiles/dodaf_minimal_strict/0.1.0/manifest.yaml
- profiles/dodaf_minimal_mixed/0.1.0/manifest.yaml
- tests/ontology_runtime/test_dodaf_minimal.py
- tests/integration/test_dodaf_minimal_cli.py
- notebooks/13_dodaf_minimal_second_pack.ipynb

notes:
- One pack now supports strict and mixed profiles without core branching.

as_json:
{
  "phase_id": "7",
  "title": "Domain Pack Generalization",
  "status": "completed",
  "confidence": "high",
  "goal": "Prove the same architecture supports a second real domain pack through local pack and profile data.",
  "why_now": [],
  "build": [],
  "acceptance": [],
  "evidence": [
    "ontology_packs/dodaf_minimal/0.1.0/manifest.yaml",
    "profiles/dodaf_minimal_strict/0.1.0/manifest.yaml",
    "prof

In [10]:
proof_summary = {
    "pack_id": "dodaf_minimal",
    "profiles": {
        "strict": "dodaf_minimal_strict",
        "mixed": "dodaf_minimal_mixed",
    },
    "proof_flow": [
        "load local pack through the same configured search roots as donor packs",
        "validate a known assertion against the strict profile",
        "route an unknown predicate through the mixed profile into a proposal",
        "review the proposal and apply the overlay without core branching",
        "prove the same path through the CLI and notebook surfaces",
    ],
    "evidence": [
        "tests/ontology_runtime/test_dodaf_minimal.py",
        "tests/integration/test_dodaf_minimal_cli.py",
        "notebooks/13_dodaf_minimal_second_pack.ipynb",
    ],
}
heading("Proof Summary")
print(json.dumps(proof_summary, indent=2))



Proof Summary
{
  "pack_id": "dodaf_minimal",
  "profiles": {
    "strict": "dodaf_minimal_strict",
    "mixed": "dodaf_minimal_mixed"
  },
  "proof_flow": [
    "load local pack through the same configured search roots as donor packs",
    "validate a known assertion against the strict profile",
    "route an unknown predicate through the mixed profile into a proposal",
    "review the proposal and apply the overlay without core branching",
    "prove the same path through the CLI and notebook surfaces"
  ],
  "evidence": [
    "tests/ontology_runtime/test_dodaf_minimal.py",
    "tests/integration/test_dodaf_minimal_cli.py",
    "notebooks/13_dodaf_minimal_second_pack.ipynb"
  ]
}


## Phase 8 Retrospective

In [11]:
phase = next(item for item in completed_phases if item.phase_id == '8')
show_phase(phase)

print("\nas_json:")
print(json.dumps(asdict(phase), indent=2))



Phase 8: Artifact Lineage Recovery
status: completed
confidence: high
goal: Recover artifact-backed provenance through a narrow three-kind model without rebuilding a fused runtime.

acceptance evidence:
- src/onto_canon6/artifacts/
- src/onto_canon6/surfaces/lineage_report.py
- tests/artifacts/test_artifact_service.py
- notebooks/14_artifact_lineage_slice.ipynb
- docs/adr/0008-start-artifact-lineage-with-a-narrow-three-kind-model-and-candidate-centered-links.md

notes:
- Artifact lineage now stays bounded, candidate-centered, and traversal-based first.

as_json:
{
  "phase_id": "8",
  "title": "Artifact Lineage Recovery",
  "status": "completed",
  "confidence": "high",
  "goal": "Recover artifact-backed provenance through a narrow three-kind model without rebuilding a fused runtime.",
  "why_now": [],
  "build": [],
  "acceptance": [],
  "evidence": [
    "src/onto_canon6/artifacts/",
    "src/onto_canon6/surfaces/lineage_report.py",
    "tests/artifacts/test_artifact_service.py",
  

In [12]:
proof_summary = {
    "artifact_kinds": ["source", "derived_dataset", "analysis_result"],
    "link_target": "candidate_assertion",
    "accepted_assertion_view": "derived by traversal/reporting",
    "evidence": [
        "tests/artifacts/test_artifact_service.py",
        "notebooks/14_artifact_lineage_slice.ipynb",
        "docs/adr/0008-start-artifact-lineage-with-a-narrow-three-kind-model-and-candidate-centered-links.md",
    ],
}
heading("Proof Summary")
print(json.dumps(proof_summary, indent=2))



Proof Summary
{
  "artifact_kinds": [
    "source",
    "derived_dataset",
    "analysis_result"
  ],
  "link_target": "candidate_assertion",
  "accepted_assertion_view": "derived by traversal/reporting",
  "evidence": [
    "tests/artifacts/test_artifact_service.py",
    "notebooks/14_artifact_lineage_slice.ipynb",
    "docs/adr/0008-start-artifact-lineage-with-a-narrow-three-kind-model-and-candidate-centered-links.md"
  ]
}


## Phase 9 Retrospective

In [13]:
phase = next(item for item in completed_phases if item.phase_id == '9')
show_phase(phase)

print("\nas_json:")
print(json.dumps(asdict(phase), indent=2))



Phase 9: Epistemic Extension
status: completed
confidence: high
goal: Add extension-local confidence and supersession over accepted candidates without changing the base review schema.

acceptance evidence:
- src/onto_canon6/extensions/epistemic/
- src/onto_canon6/surfaces/epistemic_report.py
- tests/extensions/test_epistemic_service.py
- notebooks/15_epistemic_extension_slice.ipynb
- docs/adr/0009-start-epistemic-extension-with-confidence-and-supersession-over-accepted-candidates.md

as_json:
{
  "phase_id": "9",
  "title": "Epistemic Extension",
  "status": "completed",
  "confidence": "high",
  "goal": "Add extension-local confidence and supersession over accepted candidates without changing the base review schema.",
  "why_now": [],
  "build": [],
  "acceptance": [],
  "evidence": [
    "src/onto_canon6/extensions/epistemic/",
    "src/onto_canon6/surfaces/epistemic_report.py",
    "tests/extensions/test_epistemic_service.py",
    "notebooks/15_epistemic_extension_slice.ipynb",
   

In [14]:
proof_summary = {
    "operators": ["confidence", "supersession"],
    "attachment_point": "accepted candidate assertions",
    "confidence_source_proof": "user-entered",
    "evidence": [
        "tests/extensions/test_epistemic_service.py",
        "notebooks/15_epistemic_extension_slice.ipynb",
        "docs/adr/0009-start-epistemic-extension-with-confidence-and-supersession-over-accepted-candidates.md",
    ],
}
heading("Proof Summary")
print(json.dumps(proof_summary, indent=2))



Proof Summary
{
  "operators": [
    "confidence",
    "supersession"
  ],
  "attachment_point": "accepted candidate assertions",
  "confidence_source_proof": "user-entered",
  "evidence": [
    "tests/extensions/test_epistemic_service.py",
    "notebooks/15_epistemic_extension_slice.ipynb",
    "docs/adr/0009-start-epistemic-extension-with-confidence-and-supersession-over-accepted-candidates.md"
  ]
}


## Phase 10 Completion Detail

In [15]:
phase = next(item for item in completed_phases if item.phase_id == '10')
show_phase(phase)

print('\nas_json:')
print(json.dumps(asdict(phase), indent=2))



Phase 10: Product-Facing Workflow Integration
status: completed
confidence: high
goal: Export a CLI-driven governed bundle so the successor ends in a real downstream artifact.

acceptance evidence:
- src/onto_canon6/surfaces/governed_bundle.py
- src/onto_canon6/cli.py
- tests/surfaces/test_governed_bundle.py
- tests/integration/test_cli_flow.py
- notebooks/16_governed_bundle_workflow.ipynb
- docs/adr/0010-choose-cli-driven-governed-bundle-export-as-the-first-product-facing-workflow.md

notes:
- The first outward-facing boundary stays CLI-backed and exports a governed JSON bundle instead of adding MCP too early.

as_json:
{
  "phase_id": "10",
  "title": "Product-Facing Workflow Integration",
  "status": "completed",
  "confidence": "high",
  "goal": "Export a CLI-driven governed bundle so the successor ends in a real downstream artifact.",
  "why_now": [],
  "build": [],
  "acceptance": [],
  "evidence": [
    "src/onto_canon6/surfaces/governed_bundle.py",
    "src/onto_canon6/cli.py"

In [16]:
implemented_workflow = textwrap.dedent(r"""
# Implemented product-facing slice:
# 1. ingest one source text file through the CLI
# 2. extract candidate assertions
# 3. review candidates and ontology proposals
# 4. apply ontology additions where approved
# 5. export one governed JSON bundle with provenance and governance state

workflow_demo = {
    "input": "real source text file",
    "surface": "CLI-backed governed bundle export",
    "outputs": [
        "accepted candidate assertions",
        "linked proposals and overlay applications",
        "source provenance and evidence spans",
        "candidate artifact lineage when present",
        "candidate epistemic state when present",
        "summary counts for downstream inspection",
    ],
}
""").strip()
heading("Implemented Workflow")
print(implemented_workflow)



Implemented Workflow
# Implemented product-facing slice:
# 1. ingest one source text file through the CLI
# 2. extract candidate assertions
# 3. review candidates and ontology proposals
# 4. apply ontology additions where approved
# 5. export one governed JSON bundle with provenance and governance state

workflow_demo = {
    "input": "real source text file",
    "surface": "CLI-backed governed bundle export",
    "outputs": [
        "accepted candidate assertions",
        "linked proposals and overlay applications",
        "source provenance and evidence spans",
        "candidate artifact lineage when present",
        "candidate epistemic state when present",
        "summary counts for downstream inspection",
    ],
}


## Phase 11-13 Completion Detail

In [17]:
phase_11_to_13_completion = {
    'phase_11': {
        'title': 'Canonical Graph Recovery',
        'evidence': [
            'tests/core/test_graph_service.py',
            'tests/integration/test_graph_cli.py',
            'notebooks/17_canonical_graph_recovery_slice.ipynb',
            'docs/adr/0012-start-canonical-graph-recovery-with-explicit-promotion-from-accepted-candidates.md',
        ],
    },
    'phase_12': {
        'title': 'Stable Identity and External References',
        'evidence': [
            'tests/core/test_identity_service.py',
            'tests/integration/test_identity_cli.py',
            'notebooks/18_stable_identity_slice.ipynb',
            'docs/adr/0013-start-stable-identity-with-promoted-entity-identities-alias-membership-and-explicit-external-reference-state.md',
        ],
    },
    'phase_13': {
        'title': 'Semantic Canonicalization Stack Recovery Or Replacement',
        'evidence': [
            'tests/core/test_semantic_service.py',
            'tests/integration/test_semantic_cli.py',
            'notebooks/19_semantic_canonicalization_slice.ipynb',
            'docs/adr/0014-replace-the-v1-semantic-stack-with-pack-driven-canonicalization-and-explicit-recanonicalization.md',
        ],
    },
}
heading('Phase 11-13 Completion')
print(json.dumps(phase_11_to_13_completion, indent=2))



Phase 11-13 Completion
{
  "phase_11": {
    "title": "Canonical Graph Recovery",
    "evidence": [
      "tests/core/test_graph_service.py",
      "tests/integration/test_graph_cli.py",
      "notebooks/17_canonical_graph_recovery_slice.ipynb",
      "docs/adr/0012-start-canonical-graph-recovery-with-explicit-promotion-from-accepted-candidates.md"
    ]
  },
  "phase_12": {
    "title": "Stable Identity and External References",
    "evidence": [
      "tests/core/test_identity_service.py",
      "tests/integration/test_identity_cli.py",
      "notebooks/18_stable_identity_slice.ipynb",
      "docs/adr/0013-start-stable-identity-with-promoted-entity-identities-alias-membership-and-explicit-external-reference-state.md"
    ]
  },
  "phase_13": {
    "title": "Semantic Canonicalization Stack Recovery Or Replacement",
    "evidence": [
      "tests/core/test_semantic_service.py",
      "tests/integration/test_semantic_cli.py",
      "notebooks/19_semantic_canonicalization_slice.ipynb",


## Remaining Decision Gates

In [18]:
decision_gates = {
    "benchmark_follow_on": [
        "How large should the benchmark corpus become before we treat its aggregate numbers as stronger evidence?",
        "Should structurally invalid but text-supported candidates get a dedicated remediation report before broader benchmark growth?",
    ],
    "post_phase_13": [
        "When is there enough real consumer pressure to justify an MCP surface over the governed bundle export?",
        "Which next workflow or consumer should drive any roadmap extension beyond Phase 13?",
    ],
}

heading("Remaining Decision Gates")
print(json.dumps(decision_gates, indent=2))



Remaining Decision Gates
{
  "benchmark_follow_on": [
    "How large should the benchmark corpus become before we treat its aggregate numbers as stronger evidence?",
    "Should structurally invalid but text-supported candidates get a dedicated remediation report before broader benchmark growth?"
  ],
  "post_phase_13": [
    "When is there enough real consumer pressure to justify an MCP surface over the governed bundle export?",
    "Which next workflow or consumer should drive any roadmap extension beyond Phase 13?"
  ]
}
